# Notebook 04: Cranial Component Assembly & Whole-Skull Validation
**Project**: Stegoceras Biomechanics & Uncertainty Quantification  
**Specimen**: *Stegoceras validum* (UALVP 2, referred specimen)  
**Deliverable**: Phase 2 Assembly Validation & Coordinate Harmonization  

### Objective
Construct the multi-part cranial assembly from 32 individual element meshes without applying arbitrary transformations. Perform exhaustive quantitative comparison against the deposited Whole-Skull STL (MorphoSource Media ), evaluating coordinate system congruence, sampled bidirectional Hausdorff approximations, component containment, and bilateral symmetry.


In [ ]:
import pandas as pd
import numpy as np
import trimesh
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from stegoceras_biomechanics.geometry.assembly import (
    load_all_component_meshes,
    assemble_components,
    compare_assembly_with_whole_skull,
    evaluate_component_containment_and_proximity,
    evaluate_bilateral_symmetry
)

project_root = Path('..')
components_dir = project_root / 'data' / 'meshes' / 'original' / 'components'
skull_path = project_root / 'data' / 'meshes' / 'original' / 'whole_skull' / 'WitmerLab_Stegoceras_UALVP2-000018284.stl'
inventory_path = project_root / 'data' / 'metadata' / 'geometry_inventory.csv'

inventory_df = pd.read_csv(inventory_path)
skull_mesh = trimesh.load(str(skull_path), process=False)
components_dict = load_all_component_meshes(components_dir)
assembly_mesh = assemble_components(components_dict)

print('Loaded whole skull mesh and 32 component meshes.')


### 1. Quantitative Assembly vs. Whole-Skull Comparison

In [ ]:
comp_metrics = compare_assembly_with_whole_skull(assembly_mesh, skull_mesh, n_samples=50000)
for k, v in comp_metrics.items():
    print(f'{k:40s}: {v}')


### 2. Component Containment & Proximity Audit

In [ ]:
cont_df = evaluate_component_containment_and_proximity(inventory_df, components_dict, skull_mesh)
cont_df[['element_name', 'side', 'centroid_in_whole_bbox', 'bbox_in_whole_bbox', 'mean_surface_dist_to_whole', 'p95_surface_dist_to_whole']]


### 3. Bilateral Symmetry & Taphonomic Deviation Analysis (14 Pairs)

In [ ]:
sym_df = evaluate_bilateral_symmetry(inventory_df, components_dict, skull_mesh, n_samples=5000)
sym_df[['element_name', 'left_media_id', 'right_media_id', 'area_difference_percent', 'mean_symmetry_deviation', 'sampled_bidirectional_hausdorff_symmetry']]


### 4. 3D Visualizations

In [ ]:
# Display rendered assembly and symmetry mosaic
fig_dir = project_root / 'reports' / 'figures'
if (fig_dir / '02_component_assembly_render.png').exists():
    display(Image.open(fig_dir / '02_component_assembly_render.png'))
if (fig_dir / '03_assembly_whole_overlay.png').exists():
    display(Image.open(fig_dir / '03_assembly_whole_overlay.png'))
if (fig_dir / '04_bilateral_symmetry_comparison.png').exists():
    display(Image.open(fig_dir / '04_bilateral_symmetry_comparison.png'))
